# 7CS537 CW2, Notebook 05b: Held-out attack family experiment (E3)

**Student number:** 100810453

**Purpose:** repeat CW1's held-out attack family experiment with the CW2 autoencoder added, so the CW2 anomaly detector can be compared directly with the CW1 results.

**Set-up:** on the CW1 random split, each attack family with at least 20 test flows is removed from XGBoost's training in turn. XGBoost is retrained and its detection of the removed family is measured. The autoencoder and Isolation Forest never see attacks, so their saved scores from Notebook 05 are used on the same test flows. Removing a family simulates meeting an attack type for the first time ("simulated zero-day"); it is not a real zero-day attack, which may differ from every CICIDS2017 family.

**Why a separate notebook:** retraining XGBoost in the same Python process as PyTorch crashed on macOS with a segmentation fault, because both libraries load their own copy of the OpenMP threading library. This notebook does not import PyTorch; it reads the autoencoder's scores saved by Notebook 05.

**Inputs:** CW1 data, feature list and `outputs/02_unseen_attack_experiment.csv`; `cw2/data/03_splits.parquet`; `cw2/data/05_anomaly_scores.parquet`; `cw2/outputs/04_run_config.json`.

**Outputs:** `cw2/outputs/05b_*`.

**How to run:** after Notebook 05, from the repository folder: `jupyter nbconvert --to notebook --execute --inplace cw2/05b_held_out_family.ipynb`.

## 1. Setup, data and saved scores
**What:** loads the data, checks the fingerprint, and loads the splits and the anomaly scores saved by Notebook 05.

**Why the same settings as Notebook 04's reference model:** XGBoost uses the number of trees chosen by early stopping on the full random split, with no early stopping here, exactly as CW1 did. So the only difference between runs is which family is missing from training.

In [1]:
import os
import json
import hashlib
import platform

import numpy as np
import pandas as pd
import sklearn
import xgboost as xgb
from xgboost import XGBClassifier

try:
    from IPython.display import display
except ImportError:
    display = print

RANDOM_STATE = 42
TARGET_FPR = 0.01
MIN_TEST_FLOWS = 20

ROOT = ".." if os.path.exists(os.path.join("..", "outputs", "01_feature_list.txt")) else "."
DATA_FILE = os.path.join(ROOT, "data", "cicids2017_clean.parquet")
CW1_OUT = os.path.join(ROOT, "outputs")
CW2_DIR = os.path.join(ROOT, "cw2")
OUT_DIR = os.path.join(CW2_DIR, "outputs")

if os.path.exists(DATA_FILE):
    df = pd.read_parquet(DATA_FILE)
else:
    df = pd.read_pickle(DATA_FILE.replace(".parquet", ".pkl"))
for col in ["Label", "attack_family", "source_file"]:
    df[col] = df[col].astype(str)
with open(os.path.join(CW1_OUT, "01_feature_list.txt")) as fh:
    feature_cols = [line.strip() for line in fh if line.strip()]
with open(os.path.join(OUT_DIR, "03_run_config.json")) as fh:
    nb03 = json.load(fh)
with open(os.path.join(OUT_DIR, "04_run_config.json")) as fh:
    nb04 = json.load(fh)
h = pd.util.hash_pandas_object(df[["Label", "source_file"]], index=False).to_numpy()
fingerprint = hashlib.sha256(h.tobytes()).hexdigest()[:16]
assert fingerprint == nb03["data_fingerprint"], "Data differs from Notebook 03; re-run Notebook 03"

splits = pd.read_parquet(os.path.join(CW2_DIR, "data", "03_splits.parquet"))
scores = pd.read_parquet(os.path.join(CW2_DIR, "data", "05_anomaly_scores.parquet"))
assert len(scores) == len(df)
ae_score, if_score = scores["ae_error"].to_numpy(), scores["if_score"].to_numpy()

X = df[feature_cols].to_numpy(dtype=np.float32)
y = df["is_attack"].to_numpy().astype(int)
families = df["attack_family"].to_numpy()
ae_role = splits["ae_role"].astype(str).to_numpy()
role = splits["random"].astype(str).to_numpy()
r_tr, r_va, r_te = (np.where(role == r)[0] for r in ["train", "val", "test"])
ref_trees = nb04["experiments"]["random"]["trees"]
print(f"{len(df):,} flows; reference XGBoost trees: {ref_trees}; xgboost {xgb.__version__}")

2,496,544 flows; reference XGBoost trees: 231; xgboost 3.4.1


## 2. Thresholds for the anomaly detectors on this split
**What:** the 99th percentile of each detector's score on benign validation flows of the random split, excluding any flow used to train the autoencoder.

**Why:** the same benign-only rule as Notebook 05, with a 1% false alarm target and no attack labels. Some benign Monday flows in the random test set were used to train the autoencoder, so only **attack** detection is reported here; attack flows were never in its training data.

In [2]:
cal = r_va[(y[r_va] == 0) & (ae_role[r_va] != "ae_train")]
ae_thr = float(np.quantile(ae_score[cal], 1 - TARGET_FPR))
if_thr = float(np.quantile(if_score[cal], 1 - TARGET_FPR))
print(f"Calibration flows: {len(cal):,}; autoencoder threshold {ae_thr:.5f}; Isolation Forest threshold {if_thr:.5f}")

Calibration flows: 251,796; autoencoder threshold 0.76918; Isolation Forest threshold 0.66726


## 3. Remove each family from training and test on it

In [3]:
params = dict(n_estimators=ref_trees, learning_rate=0.1, max_depth=8, subsample=0.8, colsample_bytree=0.8,
              tree_method="hist", n_jobs=-1, random_state=RANDOM_STATE)
fam_te = families[r_te]
rows = []
for fam in [f for f in pd.Series(fam_te).value_counts().index if f != "Benign"]:
    target = r_te[fam_te == fam]
    if len(target) < MIN_TEST_FLOWS:
        print(f"{fam}: skipped, only {len(target)} test flows")
        continue
    keep = r_tr[families[r_tr] != fam]
    spw = float((y[keep] == 0).sum() / (y[keep] == 1).sum())
    m = XGBClassifier(scale_pos_weight=spw, **params).fit(X[keep], y[keep], verbose=False)
    rows.append({"held_out_family": fam, "test_flows": len(target),
                 "XGBoost (family held out) %": 100 * np.mean(m.predict_proba(X[target])[:, 1] >= 0.5),
                 "Autoencoder %": 100 * np.mean(ae_score[target] >= ae_thr),
                 "Isolation Forest (CW2 threshold) %": 100 * np.mean(if_score[target] >= if_thr)})
    print(f"{fam}: done")

e3 = pd.DataFrame(rows).set_index("held_out_family")
cw1 = pd.read_csv(os.path.join(CW1_OUT, "02_unseen_attack_experiment.csv"), index_col="held_out_family")
e3 = e3.join(pd.DataFrame({
    "CW1: XGBoost (family seen) %": 100 * cw1["xgb_seen_detection"],
    "CW1: XGBoost (family held out) %": 100 * cw1["xgb_unseen_detection"],
    "CW1: Isolation Forest (CW1 threshold) %": 100 * cw1["isolation_forest_detection"]}), how="left")
e3.to_csv(f"{OUT_DIR}/05b_e3_held_out_family.csv")
display(e3.round(2))

DoS: done


DDoS: done


PortScan: done


Brute Force: done


Web Attack: done


Botnet: done
Infiltration: skipped, only 6 test flows
Heartbleed: skipped, only 2 test flows


,test_flows,XGBoost (family held out) %,Autoencoder %,Isolation Forest (CW2 threshold) %,CW1: XGBoost (family seen) %,CW1: XGBoost (family held out) %,CW1: Isolation Forest (CW1 threshold) %
held_out_family,,,,,,,
DoS,29036,4.59,28.02,18.04,99.98,10.08,87.56
DDoS,19202,63.04,7.62,2.70,99.98,63.05,62.00
PortScan,13519,0.35,0.00,0.00,99.99,0.24,0.30
Brute Force,1373,0.22,0.00,0.00,100.00,0.22,0.00
Web Attack,321,84.42,0.00,0.00,98.13,83.18,6.54
Botnet,292,0.00,0.00,0.00,87.33,0.00,1.03


## 4. Save the run configuration

In [4]:
config = {"notebook": "05b_held_out_family", "data_fingerprint": fingerprint, "target_fpr": TARGET_FPR,
          "xgb_trees": ref_trees, "ae_threshold": ae_thr, "if_threshold": if_thr, "calibration_flows": int(len(cal)),
          "versions": {"python": platform.python_version(), "xgboost": xgb.__version__,
                       "sklearn": sklearn.__version__, "pandas": pd.__version__, "numpy": np.__version__}}
with open(f"{OUT_DIR}/05b_run_config.json", "w") as fh:
    json.dump(config, fh, indent=2)
print(json.dumps(config, indent=2))

{
  "notebook": "05b_held_out_family",
  "data_fingerprint": "cb8a0f0b8bd9cf88",
  "target_fpr": 0.01,
  "xgb_trees": 231,
  "ae_threshold": 0.7691847085952759,
  "if_threshold": 0.6672627018684916,
  "calibration_flows": 251796,
  "versions": {
    "python": "3.13.13",
    "xgboost": "3.4.1",
    "sklearn": "1.9.1",
    "pandas": "3.0.6",
    "numpy": "2.5.4"
  }
}


## What to check after running

The "What this shows" note will be written from your real results once they are pushed. Compare the "Autoencoder %" column with "CW1: Isolation Forest" and "CW1: XGBoost (family held out)" for each family.